# 🪙 Assistente de Finanças — Ativos da Ibovespa (B3)

Projeto **CrewAI** com dois agentes que trabalham em sequência para gerar um
relatório de mercado a partir de dados reais da bolsa brasileira.

## Como o projeto está organizado

```
Usuário define os tickers
        │
        ▼
┌─────────────────────┐     usa a tool      ┌──────────────────────┐
│  Analista de Mercado │ ──────────────────▶ │  Consulta de Ativos  │
│   (coleta os dados)  │ ◀────────────────── │      da B3 (tool)    │
└─────────────────────┘   cotação/variação   └──────────────────────┘
        │
        │ context (passa os dados adiante)
        ▼
┌─────────────────────┐
│ Consultor Financeiro │  ──▶  relatório em Markdown (relatorio_ibovespa.md)
│  (analisa e escreve) │
└─────────────────────┘
```

## Conceitos do CrewAI que este projeto ensina
- **Tool customizada** (`BaseTool`): como dar a um agente a capacidade de agir no mundo.
- **Agent**: papel (`role`), objetivo (`goal`) e história (`backstory`).
- **Task**: o que fazer (`description`) e o formato esperado (`expected_output`).
- **context**: como o resultado de uma task vira insumo da próxima.
- **Crew + Process.sequential**: a orquestração que faz tudo rodar em ordem.

> ⚠️ **Sobre "tempo real":** o `yfinance` entrega cotações com **atraso de ~15 minutos**
> para a B3. Não é dado tick a tick — isso exigiria um feed profissional pago.
> Para estudo e relatórios, o atraso é irrelevante.

## 1. Instalação

Rode esta célula **uma vez** para instalar as dependências no ambiente.
- `crewai`: o framework de agentes (já traz o LiteLLM para falar com o Groq).
- `yfinance`: a biblioteca que busca os dados da bolsa.

In [ ]:
# Remova o '#' para instalar (rode apenas uma vez)
# !pip install crewai yfinance

## 2. Imports e chave de API

Aqui carregamos tudo o que o projeto usa e configuramos a chave do Groq.

- `getpass` pede a chave **sem exibi-la na tela** — mais seguro que escrever a chave
  direto no código (que poderia vazar se você compartilhar o notebook).
- `BaseModel` / `Field` (do Pydantic) definem o **formato de entrada** da tool.
- `BaseTool` é a classe base para criarmos nossa ferramenta customizada.
- `Agent`, `Task`, `Crew`, `Process`, `LLM` são os blocos centrais do CrewAI.

In [1]:
import os
from getpass import getpass
from typing import Type

import yfinance as yf
from pydantic import BaseModel, Field

from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import BaseTool

# --- Correção de compatibilidade com o Groq (crewai >= 1.15) ---
# O crewai marca as mensagens com a chave 'cache_breakpoint' (prompt caching
# estilo Anthropic). A API do Groq não aceita esse campo e devolve
# BadRequestError ("property 'cache_breakpoint' is unsupported"). Como a chamada
# ao Groq passa pelo LiteLLM genérico e o crewai não remove a chave nesse
# caminho, neutralizamos o marcador antes de qualquer chamada ao LLM.
import crewai.llms.cache as _crewai_cache
_crewai_cache.mark_cache_breakpoint = lambda message: message

# Pede a chave só se ela ainda não estiver no ambiente
if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("GROQ_API_KEY: ")

## 3. Configurando o LLM (Groq free-tier)

O CrewAI usa o LiteLLM por baixo dos panos, então basta o prefixo `groq/` no nome
do modelo para ele saber qual provedor chamar.

- `llama-3.3-70b-versatile` é um bom ponto de partida no plano gratuito do Groq:
  capaz o suficiente para seguir instruções e usar ferramentas.
- `temperature=0.3`: valor baixo deixa as respostas mais **objetivas e estáveis** —
  desejável para análise de dados (não queremos criatividade inventando números).

In [2]:
from crewai import Agent, Task, Crew, Process, LLM
from crewai.llms.cache import CACHE_BREAKPOINT_KEY

class GroqLLM(LLM):
    """Remove o marcador interno 'cache_breakpoint', que o Groq rejeita."""
    def _format_messages_for_provider(self, messages):
        messages = [{k: v for k, v in m.items() if k != CACHE_BREAKPOINT_KEY} for m in messages]
        return super()._format_messages_for_provider(messages)

llm = GroqLLM(
        model="groq/openai/gpt-oss-20b",
        temperature=0.7,
    )

## 4. A Tool customizada — o coração do projeto

Esta é a parte mais importante e o conceito novo: uma **ferramenta** que o agente
pode chamar para buscar dados reais. Toda tool customizada do CrewAI tem três partes:

1. **Schema de entrada** (`ConsultaAtivosInput`): uma classe Pydantic que descreve
   quais argumentos a tool recebe. A `description` de cada campo ajuda o agente a
   preencher corretamente.
2. **A tool** (`ConsultaAtivosB3`): herda de `BaseTool` e define:
   - `name`: nome curto da ferramenta.
   - `description`: **o agente lê isto para decidir quando usar a tool** — por isso é
     tão importante quanto o código. Uma descrição vaga faz o agente usar a ferramenta
     na hora errada.
   - `args_schema`: liga a tool ao schema de entrada.
3. **O método `_run`**: a lógica de fato. Aqui usamos o `yfinance`.

### Decisões de implementação
- **Sufixo `.SA`**: tickers da B3 no Yahoo Finance levam `.SA` (ex.: `PETR4.SA`).
- **Índice Ibovespa**: o símbolo é `^BVSP` e **não** leva `.SA` — tratamos esse caso.
- **Vários tickers de uma vez**: a tool recebe `"PETR4,VALE3,..."` para o agente
  buscar tudo numa única chamada. Isso evita que ele fique chamando a ferramenta em
  loop, o que multiplicaria as requisições ao LLM e estouraria o rate limit do Groq.
- **Variação diária via `history(period="5d")`**: comparamos o último fechamento com
  o anterior. É mais confiável que `.info`, que às vezes volta vazio.

In [3]:
class ConsultaAtivosInput(BaseModel):
    tickers: str = Field(
        ...,
        description="Tickers da B3 separados por vírgula, sem o sufixo .SA. "
                    "Ex: 'PETR4,VALE3,ITUB4'. Use '^BVSP' para o índice Ibovespa.",
    )


class ConsultaAtivosB3(BaseTool):
    name: str = "Consulta de Ativos da B3"
    description: str = (
        "Busca cotação atual, variação diária (%) e volume de ativos da B3/Ibovespa. "
        "Use sempre que precisar de dados de mercado de ações brasileiras."
    )
    args_schema: Type[BaseModel] = ConsultaAtivosInput

    def _run(self, tickers: str) -> str:
        linhas = []
        for bruto in tickers.split(","):
            symbol = bruto.strip().upper()
            if not symbol:
                continue
            # O índice (^BVSP) e índices em geral não levam o sufixo .SA
            yf_symbol = symbol if symbol.startswith("^") else f"{symbol}.SA"
            try:
                ativo = yf.Ticker(yf_symbol)
                hist = ativo.history(period="5d")
                if hist.empty:
                    linhas.append(f"{symbol}: dados não encontrados")
                    continue
                fechamento = hist["Close"].iloc[-1]
                anterior = hist["Close"].iloc[-2] if len(hist) >= 2 else fechamento
                variacao = (fechamento - anterior) / anterior * 100
                volume = int(hist["Volume"].iloc[-1])
                volume_fmt = f"{volume:,}".replace(",", ".")
                linhas.append(
                    f"{symbol}: R$ {fechamento:.2f} | "
                    f"variação {variacao:+.2f}% | volume {volume_fmt}"
                )
            except Exception as e:
                linhas.append(f"{symbol}: erro ao consultar ({e})")
        return "\n".join(linhas) if linhas else "Nenhum ticker válido informado."


consulta_tool = ConsultaAtivosB3()

## 5. Testando a tool isoladamente

**Boa prática:** antes de plugar a ferramenta num agente, teste-a sozinha. Se ela
não funcionar aqui, o problema não está no agente nem no LLM — está na tool.

`consulta_tool.run(...)` chama o método `_run` por baixo. Se aparecer cotação,
variação e volume, a ferramenta está OK.

In [4]:
print(consulta_tool.run(tickers="PETR4,VALE3,^BVSP"))

PETR4: R$ 45.36 | variação -1.28% | volume 47.069.700
VALE3: R$ 64.04 | variação +2.19% | volume 19.016.100
^BVSP: R$ 168498.45 | variação +1.39% | volume 0


## 6. Definindo os Agentes

Um agente é definido por três campos que moldam seu comportamento:
- **`role`**: o papel que ele desempenha.
- **`goal`**: o objetivo que ele persegue em cada execução.
- **`backstory`**: o contexto/personalidade — influencia bastante o tom e o cuidado
  das respostas.

Usamos **dois agentes** para separar responsabilidades:
- O **Analista de Mercado** recebe a `tool` (é o único que busca dados).
- O **Consultor Financeiro** não tem tool: o trabalho dele é interpretar e escrever.

In [5]:
analista_mercado = Agent(
    role="Analista de Mercado",
    goal="Coletar dados atualizados dos ativos solicitados da B3.",
    backstory=(
        "Você é um analista de mercado especializado na bolsa brasileira. "
        "É meticuloso e baseia tudo em dados reais obtidos pela ferramenta, nunca em achismos."
    ),
    tools=[consulta_tool],   # só este agente tem acesso à ferramenta
    llm=llm,
    verbose=True,
)

consultor = Agent(
    role="Consultor Financeiro",
    goal="Interpretar os dados de mercado e produzir um resumo claro e útil.",
    backstory=(
        "Você traduz números de mercado em observações compreensíveis para investidores "
        "iniciantes, com linguagem clara e sem jargão excessivo."
    ),
    llm=llm,
    verbose=True,
)

## 7. Definindo as Tasks

Cada task descreve **uma unidade de trabalho**. Os campos-chave:
- **`description`**: a instrução detalhada do que fazer.
- **`expected_output`**: o formato/resultado esperado — guia o agente a entregar o
  que você precisa.
- **`agent`**: quem executa a task.
- **`context=[coleta]`**: faz a segunda task **receber o resultado da primeira**. É
  assim que os dados coletados chegam ao consultor.
- **`output_file`**: salva o resultado final em disco automaticamente.

Repare que embutimos na descrição da análise a **obrigação de incluir o aviso** de
que o conteúdo não é recomendação de investimento. Manter isso é importante em
qualquer evolução do projeto.

In [6]:
ATIVOS = "PETR4, VALE3, ITUB4, BBDC4, ABEV3"

coleta = Task(
    description=(
        f"Consulte os dados de mercado dos seguintes ativos da B3: {ATIVOS}. "
        "Use a ferramenta de consulta uma única vez, passando todos os tickers de uma vez."
    ),
    expected_output="Uma lista com cotação, variação diária e volume de cada ativo.",
    agent=analista_mercado,
)

analise = Task(
    description=(
        "Com base nos dados coletados, escreva um resumo de mercado em português. "
        "Destaque os ativos com maior alta e maior baixa do dia, comente o volume e "
        "aponte qualquer movimento que chame atenção. "
        "Inclua OBRIGATORIAMENTE ao final o aviso: 'Este conteúdo é apenas informativo "
        "e não constitui recomendação de investimento.'"
    ),
    expected_output=(
        "Relatório em Markdown com: (1) tabela resumo dos ativos, (2) destaques de alta "
        "e baixa, (3) observação sobre volume, (4) aviso de que não é recomendação."
    ),
    agent=consultor,
    context=[coleta],                      # recebe o resultado da task 'coleta'
    output_file="relatorio_ibovespa.md",   # salva o relatório em disco
)

## 8. Montando a Crew

A `Crew` junta agentes e tasks e define **como** eles rodam.
`Process.sequential` executa as tasks na ordem da lista: primeiro a `coleta`,
depois a `analise`. (Existe também o `Process.hierarchical`, com um agente gerente
que delega — fica para uma aula futura.)

In [7]:
crew = Crew(
    agents=[analista_mercado, consultor],
    tasks=[coleta, analise],
    process=Process.sequential,
    verbose=True,
)

## 9. Executando

No **Jupyter** já existe um event loop rodando, então `crew.kickoff()` levanta um
`RuntimeError`. A versão assíncrona `await crew.kickoff_async()` resolve isso.

> 💡 Em um **script normal** (`.py`) ou no **Streamlit**, use `crew.kickoff()` mesmo —
> o `kickoff_async()` só é necessário aqui no notebook.

In [8]:
resultado = await crew.kickoff_async()
print(resultado)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: f3ab0747-6f5c-4eb0-9413-f299bd2237ed                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Consulte os dados de mercado dos seguintes ativos da B3: PETR4, VALE3, ITUB4, BBDC4, ABEV3. Use a        │
│  ferramenta de consulta uma única vez, passando todos os tickers de uma vez.                                    │
│  ID: 16ed10b9-0204-4f5c-93d5-14b5f1efdd63                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Analista de Mercado                                                                                     │
│                                                                                                                 │
│  Task: Consulte os dados de mercado dos seguintes ativos da B3: PETR4, VALE3, ITUB4, BBDC4, ABEV3. Use a        │
│  ferramenta de consulta uma única vez, passando todos os tickers de uma vez.                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: consulta_de_ativos_da_b3                                                                                 │
│  Args: {'tickers': 'PETR4,VALE3,ITUB4,BBDC4,ABEV3'}                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────── ✅ Tool Execution Completed (#1) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: consulta_de_ativos_da_b3                                                                                 │
│  Output: PETR4: R$ 45.36 | variação -1.28% | volume 47.069.700                                                  │
│  VALE3: R$ 64.04 | variação +2.19% | volume 19.016.100                                                          │
│  ITUB4: R$ 40.53 | variação +1.40% | volume 27.992.000                                                          │
│  BBDC4: R$ 14.20 | variação +3.12% | volume 45.442.800                                                          │
│  ABEV3: R$ 15.48 | variação +0.00% | volume 29.624.200                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


... (5 outputs left out)

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Consultor Financeiro                                                                                    │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Resumo de Mercado – 06/11/2026                                                                               │
│                                                                                                                 │
│  | Ativo | Preço (R$) | Variação | Volume |                                                                     │
│  |-------|------------|----------|--------|                                                                     │
│  | PETR4 | 45,36 | **‑1,28 %** | 47 069 700 |                                                                   │
│  | VALE3 | 64,04 | +2,19 % | 19 016 100 |                                                                       │
│  | ITUB4 | 40,53 | +1,40 % | 27 992 000 |                                                                       │
│  | BBDC4 | 14,20 | **+3,12 %** | 45 442 800 |                                                                   │
│  | ABEV3 | 15,48 | 0,00 % | 29 624 200 |                                                                        │
│                                                                                                                 │
│  ## Destaques do Dia                                                                                            │
│  - **Maior alta:** **BBDC4** (+3,12 %) – crescimento robusto em relação ao fechamento anterior.                 │
│  - **Maior baixa:** **PETR4** (‑1,28 %) – queda moderada, mas ainda o maior declínio do portfólio.              │
│                                                                                                                 │
│  ## Observação sobre Volume                                                                                     │
│  - O volume mais alto registrou **PETR4** (47,069,700 ações), indicando forte liquidez apesar da queda.         │
│  - **BBDC4** também apresentou volume elevado (45,442,800), reforçando a relevância da alta observada.          │
│  - Os demais ativos tiveram volumes mais moderados, com **VALE3** e **ITUB4** entre os mais negociados no       │
│  grupo.                                                                                                         │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  > Este conteúdo é apenas informativo e não constitui recomendação de investimento.                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Com base nos dados coletados, escreva um resumo de mercado em português. Destaque os ativos com maior    │
│  alta e maior baixa do dia, comente o volume e aponte qualquer movimento que chame atenção. Inclua              │
│  OBRIGATORIAMENTE ao final o aviso: 'Este conteúdo é apenas informativo e não constitui recomendação de         │
│  investimento.'                                                                                                 │
│  Agent: Consultor Financeiro                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: f3ab0747-6f5c-4eb0-9413-f299bd2237ed                                                                       │
│  Final Output: # Resumo de Mercado – 06/11/2026                                                                 │
│                                                                                                                 │
│  | Ativo | Preço (R$) | Variação | Volume |                                                                     │
│  |-------|------------|----------|--------|                                                                     │
│  | PETR4 | 45,36 | **‑1,28 %** | 47 069 700 |                                                                   │
│  | VALE3 | 64,04 | +2,19 % | 19 016 100 |                                                                       │
│  | ITUB4 | 40,53 | +1,40 % | 27 992 000 |                                                                       │
│  | BBDC4 | 14,20 | **+3,12 %** | 45 442 800 |                                                                   │
│  | ABEV3 | 15,48 | 0,00 % | 29 624 200 |                                                                        │
│                                                                                                                 │
│  ## Destaques do Dia                                                                                            │
│  - **Maior alta:** **BBDC4** (+3,12 %) – crescimento robusto em relação ao fechamento anterior.                 │
│  - **Maior baixa:** **PETR4** (‑1,28 %) – queda moderada, mas ainda o maior declínio do portfólio.              │
│                                                                                                                 │
│  ## Observação sobre Volume                                                                                     │
│  - O volume mais alto registrou **PETR4** (47,069,700 ações), indicando forte liquidez apesar da queda.         │
│  - **BBDC4** também apresentou volume elevado (45,442,800), reforçando a relevância da alta observada.          │
│  - Os demais ativos tiveram volumes mais moderados, com **VALE3** e **ITUB4** entre os mais negociados no       │
│  grupo.                                                                                                         │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  > Este conteúdo é apenas informativo e não constitui recomendação de investimento.                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

# Resumo de Mercado – 06/11/2026

| Ativo | Preço (R$) | Variação | Volume |
|-------|------------|----------|--------|
| PETR4 | 45,36 | **‑1,28 %** | 47 069 700 |
| VALE3 | 64,04 | +2,19 % | 19 016 100 |
| ITUB4 | 40,53 | +1,40 % | 27 992 000 |
| BBDC4 | 14,20 | **+3,12 %** | 45 442 800 |
| ABEV3 | 15,48 | 0,00 % | 29 624 200 |

## Destaques do Dia
- **Maior alta:** **BBDC4** (+3,12 %) – crescimento robusto em relação ao fechamento anterior.  
- **Maior baixa:** **PETR4** (‑1,28 %) – queda moderada, mas ainda o maior declínio do portfólio.

## Observação sobre Volume
- O volume mais alto registrou **PETR4** (47,069,700 ações), indicando forte liquidez apesar da queda.  
- **BBDC4** também apresentou volume elevado (45,442,800), reforçando a relevância da alta observada.  
- Os demais ativos tiveram volumes mais moderados, com **VALE3** e **ITUB4** entre os mais negociados no grupo.

---

> Este conteúdo é apenas informativo e não constitui recomendação de investimento.


## 10. Próximos passos

- **Trocar a fonte de dados** por `brapi.dev` (API brasileira com token gratuito) para
  acessar dados fundamentalistas (P/L, dividend yield).
- **Adicionar um terceiro agente** de gestão de risco, comentando volatilidade.
- **Estruturar a saída** com `output_pydantic` em vez de texto livre.
- **Transformar em chatbot** com Streamlit (veja o arquivo `streamlit_app.py`).